# Reproducing SUPREME experiments

This notebook is a runnable companion to the README's **Reproducing the paper**
section and [`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md).
It walks through installing the framework, running a quick smoke test, launching
the paper's experiment grid, rendering the result tables, and extending the
framework via the registry.

> **Hardware.** The unlearning pipeline needs a GPU (NVIDIA CUDA, or Apple
> Silicon MPS) for realistic runtimes. The paper's numbers were produced on a
> **single NVIDIA L40S (48 GB)**; multi-GPU runs are not bit-equivalent to
> single-GPU runs because of distributed gradient averaging, so reproduce the
> paper single-GPU. The shell cells below assume you run this notebook from a
> clone of the repository with the environment activated.

## 1. Install

Clone the repo and install the framework. For an exact reproduction of the
reported numbers, check out the tagged reference release **`v0.1.0-paper`**
first - later commits keep behaviour, defaults and seeds identical, but the tag
is the guaranteed reference point.

In [ ]:
# One-time setup - run this first. The Makefile is the single entry point for all
# environment setup in this repo (venv + pinned deps + editable install + git
# hook), so we invoke it here rather than calling pip directly.
#
# In a shell you would first clone (and optionally pin the paper tag):
#   git clone https://github.com/pedroandreou/supreme-unlearning.git
#   cd supreme-unlearning && git checkout v0.1.0-paper   # optional
import subprocess, platform, pathlib

# Find the repo root (holds the Makefile/pyproject.toml) from the kernel's cwd.
root = pathlib.Path.cwd()
while root != root.parent and not (root / 'Makefile').exists():
    root = root.parent
assert (root / 'Makefile').exists(), f'repo root not found from {pathlib.Path.cwd()}'

# Linux + NVIDIA -> `make cuda`; Apple Silicon / CPU / other -> `make mps`.
target = 'cuda' if platform.system() == 'Linux' else 'mps'
print('repo root:', root, '| running: make', target)

# ON_EXISTING=reuse keeps it non-interactive (reinstall into the existing venv).
subprocess.run(['make', target, 'ON_EXISTING=reuse'], cwd=root, check=True)
print("done. The Makefile installs into the venv (default: unlearning). Make sure")
print("THIS notebook's kernel is that venv (restart + pick it from the selector).")

## 2. Configure credentials

Evaluation results are logged **exclusively to Weights & Biases**, and ViT
weights are pulled from the Hugging Face Hub, so both tokens are required. Copy
the template and fill in your keys (see [`docs/environment_setup.md`](../docs/environment_setup.md)).

In [ ]:
# Load credentials from the repo-root .env. First time: copy and fill it in:
#   cp .env.example .env   # then set WANDB_KEY, WANDB_USERNAME, HUGGING_FACE_HUB_TOKEN
from dotenv import load_dotenv
import os, pathlib

# Resolve the repo root so this works regardless of the kernel's cwd.
_root = globals().get('root')
if _root is None:
    _root = pathlib.Path.cwd()
    while _root != _root.parent and not (_root / 'pyproject.toml').exists():
        _root = _root.parent
load_dotenv(_root / '.env', override=True)

# Check the names the framework + .env.example actually use (NOT WANDB_API_KEY / HF_TOKEN):
#   W&B key  -> WANDB_KEY  (the framework also accepts WANDB_API_KEY as a fallback)
#   HF token -> HUGGING_FACE_HUB_TOKEN  (auto-detected by huggingface_hub)
wandb_key = os.getenv('WANDB_KEY') or os.getenv('WANDB_API_KEY')
print('W&B key set      :', bool(wandb_key))
print('W&B username set :', bool(os.getenv('WANDB_USERNAME')))
print('HF token set     :', bool(os.getenv('HUGGING_FACE_HUB_TOKEN')))

## 3. Verify the install

`import supreme` is lightweight (it does not import the PyTorch/Lightning stack),
so it is a quick check that the package resolves and exposes its public API.

In [ ]:
import supreme

print('SUPREME version:', supreme.__version__)
print('Public API:', [n for n in supreme.__all__ if n.startswith(('register', 'run'))])

# Built-in components available out of the box:
print('Models   :', supreme.project_config.model_names)
print('Methods  :', supreme.project_config.all_methods)
print('Datasets :', supreme.project_config.dataset_names)
print('Metrics  :', supreme.project_config.evaluation_metrics)

## 4. Smoke test (one cell)

Before the full grid, confirm the end-to-end **train -> unlearn -> evaluate**
pipeline works on a single small cell. Re-running is safe: training checkpoints,
unlearning checkpoints and already-logged W&B results are detected and skipped.

Two equivalent ways to launch it:

In [ ]:
# (a) Smoke test via the experiment-grid driver. Two Jupyter quirks shape this:
#   - large cell outputs get truncated, and
#   - the `!cmd &` shell magic raises "Background processes not supported".
# So we launch with subprocess in the background and redirect the FULL output to a
# .log under logs/output_log_files/ (open it in an editor to watch it grow). start_new_session
# detaches it like nohup (survives a kernel restart). `root` comes from section 1.
#
# Runs on any OS - auto-selects MPS/CPU; the PAPER's numbers need Linux + NVIDIA GPU
# + CUDA. The pipeline's flock/etc. are handled portably, so this runs on macOS too.
import subprocess
logdir = root / "logs" / "output_log_files"
logdir.mkdir(parents=True, exist_ok=True)
smoke_log = logdir / "smoke_test.log"

cmd = ["bash", str(root / "supreme" / "run_local.sh"),
       "--gpu", "0", "--models", "ViT", "--training-seeds", "260",
       "--methods", "retrain,finetune,ssd",
       "--strategies", "random_", "--datasets", "Cifar10",
       "--forget-percs", "0.01"]
with open(smoke_log, "w") as f:
    proc = subprocess.Popen(cmd, cwd=str(root), stdout=f,
                            stderr=subprocess.STDOUT, start_new_session=True)
print(f"Smoke test launched in the background (PID {proc.pid}). Full log:")
print("  ", smoke_log)
print(f"Watch progress:  !tail -n 80 {smoke_log}")

In [ ]:
# (b) The Python API for a SINGLE unlearning stage - same arguments as the
#     `supreme-unlearn` console script.
#
#     You normally do NOT hand-write -weight_path. The driver in (a) discovers
#     the trained checkpoint for you: supreme/MAIN.sh -> find_checkpoint() calls
#     supreme/utils/unlearning/update_checkpoint_paths.py, which finds the latest
#     *-best.pth (gated on a TRAINING_DONE marker) and passes it as -weight_path
#     automatically. Use this direct call only to target one specific checkpoint.
# import supreme
# supreme.run_unlearning([
#     '-method', 'ssd', '-net', 'ViT', '-dataset', 'Cifar10',
#     '-type_of_unlearning_strategy', 'random_', '-seed', '260',
#     '-precision', '32-true', '-weight_path', '<path/to/trained.ckpt>',
# ])

## 5. Reproduce the paper's experiment grid

The command below matches the paper exactly: six unlearning methods plus the
retrain baseline, both scenarios (`fullclass`, `random_`), both architectures,
the paper's **I = 10 training seeds** (260-269), and a 0.1% forget set for the
random-sample scenario.

**Matched-seed protocol (J = K = 1, the default).** For each training seed the
unlearning and evaluation seeds are tied to it: **s_t = s_u = s_e**. It is
selected by `--unlearning-seeds "0" --evaluation-seeds "0"` (passed explicitly
below for clarity, though they are the defaults). See the seed-protocol table in
[`supreme/README.md`](../supreme/README.md) and [`docs/notation.md`](../docs/notation.md);
the decoupled protocols (J > 1 and/or K > 1) are for variance studies, not the
paper's headline numbers.

This is a long-running production job - launch it on the GPU machine and let it
populate W&B. Interruptions are safe to resume (finished cells are skipped).

Because it can run for hours, the cell below launches it with `nohup` in the
**background** and redirects all output to `logs/repro_grid.log`, so the cell
returns immediately and the notebook stays responsive (the job even survives a
kernel restart). The cell after it tails that log - re-run it anytime to check
progress.

Pins Face Recognition requires a manual Kaggle download - see
[`docs/adding_pinsfacerecognition.md`](../docs/adding_pinsfacerecognition.md).

In [ ]:
# Reproduce the paper's full grid in the BACKGROUND via subprocess (Jupyter's
# `!cmd &` magic raises "Background processes not supported"), with the FULL output
# redirected to a .log under logs/output_log_files/. start_new_session detaches it like
# nohup (survives a kernel restart). `root` comes from section 1.
#
# Matched-seed protocol (J=K=1): I=10 training seeds (260-269), and for each one
# s_t = s_u = s_e. The --unlearning-seeds/--evaluation-seeds "0" flags pin it
# explicitly (they are also the defaults).
#
# Runs on any OS, but the PAPER's numbers require Linux + NVIDIA GPU + CUDA - on a
# macOS/CPU host this trains on MPS/CPU and is far too slow for the full grid.
# PinsFaceRecognition also needs a manual Kaggle download first.
import subprocess
logdir = root / "logs" / "output_log_files"
logdir.mkdir(parents=True, exist_ok=True)
logfile = logdir / "repro_grid.log"

cmd = ["bash", str(root / "supreme" / "run_local.sh"),
       "--gpu", "0",
       "--datasets", "PinsFaceRecognition",
       "--models", "ResNet18,ViT",
       "--strategies", "fullclass,random_",
       "--methods", "retrain,finetune,bad_teacher,random_labeling,unsir,ssd,lfssd",
       "--forget-percs", "0.001",
       "--training-seeds", "260,261,262,263,264,265,266,267,268,269",
       "--unlearning-seeds", "0",
       "--evaluation-seeds", "0"]
with open(logfile, "w") as f:
    proc = subprocess.Popen(cmd, cwd=str(root), stdout=f,
                            stderr=subprocess.STDOUT, start_new_session=True)
print(f"Launched in the background (PID {proc.pid}). Log:", logfile)
print("Monitor it with the next cell (re-run anytime).")

In [ ]:
# Monitor the background grid run (re-run anytime; the job keeps going).
!tail -n 50 {logfile}

# Is it still running?   ->  !pgrep -fl run_local.sh
# Stop it early?         ->  !pkill -f run_local.sh

## 6. Render the paper tables

Once the grid has populated W&B, export the metrics and render the three paper
LaTeX tables with
[`supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb`](../supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb).
The export-to-LaTeX workflow and troubleshooting notes are in
[`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md) and
[`docs/tooling.md`](../docs/tooling.md).